In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

base = "/Volumes/northstar_network_bronze/nppes_data"

# One-line header file -> list of column names
raw_cols = spark.read.csv(f"{base}/npi_data_header/npidata_pfile_20050523-20260913_fileheader.csv").first()
cols = [re.sub(r"[ ,;{}()=\n\t]+", "_", str(c).strip()).strip("_") for c in raw_cols]

schema = StructType([StructField(c, StringType(), True) for c in cols])

(
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "false")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(schema)
    .load(f"{base}/nppes_data_storage/")
    .writeStream
    .option("checkpointLocation", f"{base}/nppes_checkpoint/")
    .trigger(availableNow=True)
    .toTable("northstar_network_bronze.nppes_data.npi_bronze")
    )